In [ ]:
# Лабораторная работа №2
# Ермакова Арина ЦИБ-251

## Описание варианта 9
# Граф  5 вершин, рёбра: (1,2), (2,3), (3,4), (4,5), (5,1)
# BST Элементы: {28, 16, 35, 12, 22, 30, 40}`
# Операции BST Найти: 22 Удалить: 28
# Heap Sort Массив: [28, 16, 35, 12, 22, 30, 40]

## 1. Описание структур данных
# Граф: хранится через список смежности для быстрого обхода. Матрицы смежности/инцидентности генерируются динамически.
# BST: реализован через рекурсивные узлы `TreeNode`. Поиск/удаление работают за O(log n) в сбалансированном случае.
# Куча: бинарная max-куча на основе массива. Индексы потомков: 2i+1, 2i+2.

## 2. Сложность алгоритмов (Big O)
# Матрица смежности: Время O(V²), Память O(V²)
# Матрица инцидентности: Время O(V·E), Память O(V·E)
# Обход графа (DFS/BFS): Время O(V + E), Память O(V) (рекурсия/очередь)
# Операции BST (вставка/поиск/удаление):
# Время: O(log n) в среднем, O(n) в худшем случае (вырожденное дерево)
# Память: O(h) на стек вызовов, где `h` — высота дерева
# Heap Sort: Время O(n log n) во всех случаях, Память O(1) (сортировка in-place)

In [ ]:
from collections import defaultdict
from typing import List, Tuple, Dict

class Graph:
    def __init__(self, vertices: List[int], edges: List[Tuple[int, int]]):
        self.vertices = sorted(vertices)
        self.edges = edges
        self.vertex_index = {v: i for i, v in enumerate(self.vertices)}
        self.adj_list = self._build_adj_list()

    def _build_adj_list(self) -> Dict[int, List[int]]:
        adj = defaultdict(list)
        for u, v in self.edges:
            adj[u].append(v)
            adj[v].append(u)
        return adj

    def adjacency_matrix(self) -> List[List[int]]:
        n = len(self.vertices)
        matrix = [[0]*n for _ in range(n)]
        for u, v in self.edges:
            i, j = self.vertex_index[u], self.vertex_index[v]
            matrix[i][j] = 1
            matrix[j][i] = 1
        return matrix

    def incidence_matrix(self) -> List[List[int]]:
        n, m = len(self.vertices), len(self.edges)
        matrix = [[0]*m for _ in range(n)]
        for idx, (u, v) in enumerate(self.edges):
            matrix[self.vertex_index[u]][idx] = 1
            matrix[self.vertex_index[v]][idx] = 1
        return matrix

    def dfs_connected_components(self) -> List[List[int]]:
        visited = set()
        components = []
        def dfs(v, comp):
            visited.add(v)
            comp.append(v)
            for neighbor in self.adj_list[v]:
                if neighbor not in visited:
                    dfs(neighbor, comp)
        for v in self.vertices:
            if v not in visited:
                comp = []
                dfs(v, comp)
                components.append(sorted(comp))
        return components

    def print_matrices(self):
        print("Матрица смежности:")
        adj = self.adjacency_matrix()
        print("   ", " ".join(f"{v:2d}" for v in self.vertices))
        for i, v in enumerate(self.vertices):
            print(f"{v:2d}:", " ".join(f"{adj[i][j]:2d}" for j in range(len(self.vertices))))

        print("\n Матрица инцидентности:")
        inc = self.incidence_matrix()
        print("   ", " ".join(f"E{i+1:2d}" for i in range(len(self.edges))))
        for i, v in enumerate(self.vertices):
            print(f"{v:2d}:", " ".join(f"{inc[i][j]:2d}" for j in range(len(self.edges))))

In [ ]:
from typing import Optional, List, Tuple

class TreeNode:
    def __init__(self, value: int):
        self.value = value
        self.left: Optional['TreeNode'] = None
        self.right: Optional['TreeNode'] = None

class BinarySearchTree:
    def __init__(self):
        self.root: Optional[TreeNode] = None

    def insert(self, value: int):
        if not self.root:
            self.root = TreeNode(value)
        else:
            self._insert(self.root, value)

    def _insert(self, node: TreeNode, value: int):
        if value < node.value:
            if not node.left: node.left = TreeNode(value)
            else: self._insert(node.left, value)
        elif value > node.value:
            if not node.right: node.right = TreeNode(value)
            else: self._insert(node.right, value)

    def search(self, value: int) -> bool:
        return self._search(self.root, value)

    def _search(self, node: Optional[TreeNode], value: int) -> bool:
        if not node: return False
        if value == node.value: return True
        return self._search(node.left, value) if value < node.value else self._search(node.right, value)

    def delete(self, value: int) -> bool:
        self.root, deleted = self._delete(self.root, value)
        return deleted

    def _delete(self, node: Optional[TreeNode], value: int) -> Tuple[Optional[TreeNode], bool]:
        if not node: return None, False
        if value < node.value:
            node.left, deleted = self._delete(node.left, value)
            return node, deleted
        elif value > node.value:
            node.right, deleted = self._delete(node.right, value)
            return node, deleted
        else:
            if not node.left: return node.right, True
            if not node.right: return node.left, True
            min_node = self._find_min(node.right)
            node.value = min_node.value
            node.right, _ = self._delete(node.right, min_node.value)
            return node, True

    def _find_min(self, node: TreeNode) -> TreeNode:
        while node.left: node = node.left
        return node

    def inorder(self) -> List[int]:
        res = []
        self._inorder(self.root, res)
        return res

    def _inorder(self, node: Optional[TreeNode], res: List[int]):
        if node:
            self._inorder(node.left, res)
            res.append(node.value)
            self._inorder(node.right, res)

    def print_tree(self, node: Optional[TreeNode] = None, level: int = 0, prefix: str = "Root: "):
        if node is None: node = self.root
        if node is None:
            print("Пустое дерево")
            return
        print(" " * (level * 4) + prefix + str(node.value))
        if node.left or node.right:
            if node.left:
                self.print_tree(node.left, level + 1, "L--- ")
            else:
                print(" " * ((level+1)*4) + "L--- None")
            if node.right:
                self.print_tree(node.right, level + 1, "R--- ")
            else:
                print(" " * ((level+1)*4) + "R--- None")

In [ ]:
def heapify(arr, n, i):
    largest = i
    l = 2 * i + 1
    r = 2 * i + 2
    if l < n and arr[l] > arr[largest]: largest = l
    if r < n and arr[r] > arr[largest]: largest = r
    if largest != i:
        arr[i], arr[largest] = arr[largest], arr[i]
        heapify(arr, n, largest)

def heap_sort(arr):
    n = len(arr)
    arr = arr.copy()
    for i in range(n // 2 - 1, -1, -1):
        heapify(arr, n, i)
    for i in range(n - 1, 0, -1):
        arr[0], arr[i] = arr[i], arr[0]
        heapify(arr, i, 0)
    return arr

def heap_sort_steps(arr):
    n = len(arr)
    arr = arr.copy()
    steps = [arr.copy()]
    for i in range(n // 2 - 1, -1, -1):
        heapify(arr, n, i)
    steps.append(arr.copy())
    for i in range(n - 1, 0, -1):
        arr[0], arr[i] = arr[i], arr[0]
        heapify(arr, i, 0)
        steps.append(arr.copy())
    return steps

In [ ]:
print("="*60)
print("ЧАСТЬ 1: ГРАФЫ")
print("="*60)
vertices = [1, 2, 3, 4, 5]
edges = [(1,2), (2,3), (3,4), (4,5), (5,1)]
g = Graph(vertices, edges)
g.print_matrices()
print("\n Компоненты связности (DFS):", g.dfs_connected_components())

ЧАСТЬ 1: ГРАФЫ
Матрица смежности:
     1  2  3  4  5
 1:  0  1  0  0  1
 2:  1  0  1  0  0
 3:  0  1  0  1  0
 4:  0  0  1  0  1
 5:  1  0  0  1  0

 Матрица инцидентности:
    E 1 E 2 E 3 E 4 E 5
 1:  1  0  0  0  1
 2:  1  1  0  0  0
 3:  0  1  1  0  0
 4:  0  0  1  1  0
 5:  0  0  0  1  1

 Компоненты связности (DFS): [[1, 2, 3, 4, 5]]


In [ ]:
print("\n" + "="*60)
print("ЧАСТЬ 2: БИНАРНОЕ ДЕРЕВО ПОИСКА (BST)")
print("="*60)
bst_data = [28, 16, 35, 12, 22, 30, 40]
bst = BinarySearchTree()
for v in bst_data: bst.insert(v)

print("Исходное дерево (In-order):", bst.inorder())
bst.print_tree()

print(f"\n Поиск 22: {'Найдено' if bst.search(22) else 'Не найдено'}")
print("Удаление 28...")
bst.delete(28)
print("Дерево после удаления (In-order):", bst.inorder())
bst.print_tree()


ЧАСТЬ 2: БИНАРНОЕ ДЕРЕВО ПОИСКА (BST)
Исходное дерево (In-order): [12, 16, 22, 28, 30, 35, 40]
Root: 28
    L--- 16
        L--- 12
        R--- 22
    R--- 35
        L--- 30
        R--- 40

 Поиск 22: Найдено
Удаление 28...
Дерево после удаления (In-order): [12, 16, 22, 30, 35, 40]
Root: 30
    L--- 16
        L--- 12
        R--- 22
    R--- 35
        L--- None
        R--- 40


In [ ]:
print("\n" + "="*60)
print("ЧАСТЬ 3: HEAP SORT")
print("="*60)
arr = [28, 16, 35, 12, 22, 30, 40]
print(f"Исходный массив: {arr}")
steps = heap_sort_steps(arr)
for i, s in enumerate(steps):
    print(f"   Шаг {i}: {s}")
print(f"\n Отсортированный массив: {heap_sort(arr)}")


ЧАСТЬ 3: HEAP SORT
Исходный массив: [28, 16, 35, 12, 22, 30, 40]
   Шаг 0: [28, 16, 35, 12, 22, 30, 40]
   Шаг 1: [40, 22, 35, 12, 16, 30, 28]
   Шаг 2: [35, 22, 30, 12, 16, 28, 40]
   Шаг 3: [30, 22, 28, 12, 16, 35, 40]
   Шаг 4: [28, 22, 16, 12, 30, 35, 40]
   Шаг 5: [22, 12, 16, 28, 30, 35, 40]
   Шаг 6: [16, 12, 22, 28, 30, 35, 40]
   Шаг 7: [12, 16, 22, 28, 30, 35, 40]

 Отсортированный массив: [12, 16, 22, 28, 30, 35, 40]
